In [ ]:
import pandas as pd
import numpy as np
import neurokit2 as nk
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [ ]:
mode = 1
"""
mode = 0 --> non MACE
mode = 1 --> MACE
"""
sampling_rate = 500

In [ ]:
if mode == 0:
    file_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_1_30_10s_ECG_signal\non_mace_zscore/'
elif mode == 1:
    file_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_1_30_10s_ECG_signal\mace_zscore/'

if mode == 0:
    save_dir = r'V:\dunwei\MACE\dataset\ECG_quality\non_mace'
elif mode == 1:
    save_dir = r'V:\dunwei\MACE\dataset\ECG_quality\mace'

In [ ]:
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Created directory: {save_dir}")
else:
    print(f"Directory already exists: {save_dir}")

In [ ]:
filenames = []
for basename in os.listdir(file_path):
    if basename.endswith('.npy'):
        filenames.append(basename.split('.')[0])
print(filenames)

In [ ]:
"""
Each ECG signal is 10s long, sampled at 500Hz, so each signal has 5000 data points.
Use quality method zhao2018 to check ECG signal quality (Unacceptable, Barely acceptable, Acceptable)
Use quality method averageQRS to check ECG signal quality (0-1)
Draw raw ECG signal with R-peaks marked and quality (0-1)
error id : MACE237、MACE265、MACE316、MACE328、MACE469、MACE481、MACE519, Too few peaks detected to compute the rate. Returning empty vector.
Reason : ECG signal is too short < 3 periods, so that the R-peak detection algorithm cannot detect enough R-peaks.
Non MACE time : 10hrs
MACE time : 40mins
"""

In [ ]:
error_id_window_list = []
for i, filename in enumerate(tqdm(filenames, desc="Processing files")):
    # if i > 2:
    #     break
    try:
        ECG_data = np.load(os.path.join(file_path, f'{filename}.npy'), allow_pickle=True)
        ECG_clean_signal = ECG_data[:,1:]

        ECG_window_signal_list = []
        rpeaks_info_list = []
        window_num_list = []
        quality_zhao_list = []
        quality_averageQRS_list = []
        quality_df = pd.DataFrame()

        for window in range(ECG_clean_signal.shape[0]):
            # if window > 2:
            #     break
            ECG_window_signal = ECG_clean_signal[window]
            ECG_window_signal_list.append(ECG_window_signal)
            rpeaks, info = nk.ecg_peaks(ECG_window_signal, sampling_rate=sampling_rate, method="neurokit", correct_artifacts=True, show=False)
            rpeaks_info_list.append(info['ECG_R_Peaks'])
            quality_zhao = nk.ecg_quality(ECG_window_signal, info['ECG_R_Peaks'], sampling_rate=sampling_rate, method="zhao2018")
            quality_zhao_list.append(quality_zhao)
            window_num_list.append(window+1)
            try:
                quality_averageQRS = nk.ecg_quality(ECG_window_signal, info['ECG_R_Peaks'], sampling_rate=sampling_rate, method="averageQRS")
                quality_averageQRS_normalized = (quality_averageQRS - np.mean(quality_averageQRS)) / np.std(quality_averageQRS)
                quality_averageQRS_list.append(quality_averageQRS_normalized)
            except Exception as e:
                print(f"Error computing averageQRS for file {filename}, window {window+1}: {e}")
                quality_averageQRS_list.append(np.zeros_like(ECG_window_signal))
                error_id_window_list.append({'error_id_window': f'{filename}_window{window+1}', 'Reason': str(e)})
        

        quality_df = pd.DataFrame({'window_num':window_num_list, f'{filename}': quality_zhao_list})
        if not os.path.exists(os.path.join(save_dir, filename)):
            os.makedirs(os.path.join(save_dir, filename))
        save_path = os.path.join(save_dir, filename)
        quality_df.to_csv(os.path.join(save_path, f'{filename}_quality.csv'), index=False)
        
        win_length = len(ECG_window_signal_list)
        time = np.linspace(0, ECG_clean_signal.shape[1]/sampling_rate, ECG_clean_signal.shape[1])
        for subplot in range(0, win_length, 2):
            ECG_window_signal_arr_fir = np.array(ECG_window_signal_list[subplot])
            fig, axes = plt.subplots(2, 1, figsize=(8, 8))
            axes[0].plot(time, ECG_window_signal_arr_fir, color='Tab:blue', label='ECG Signal')
            axes[0].plot(time, quality_averageQRS_list[subplot], color='red', label='Quality (averageQRS, normalized)')
            axes[0].scatter(time[rpeaks_info_list[subplot]], ECG_window_signal_arr_fir[rpeaks_info_list[subplot]], color='orange', marker='o', label='R-peaks')
            axes[0].set_title(f'ECG Signal - Window {subplot+1}')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Voltage (mV)')
            axes[0].legend(fontsize=8)
            axes[0].grid(alpha=0.3)
            
            if subplot+1 < win_length:
                ECG_window_signal_arr_sec = np.array(ECG_window_signal_list[subplot+1])
                axes[1].plot(time, ECG_window_signal_arr_sec, color='Tab:blue', label='ECG Signal')
                axes[1].plot(time, quality_averageQRS_list[subplot+1], color='red', label='Quality (averageQRS, normalized)')
                axes[1].scatter(time[rpeaks_info_list[subplot+1]], ECG_window_signal_arr_sec[rpeaks_info_list[subplot+1]], color='orange', marker='o', label='R-peaks')
                axes[1].set_title(f'ECG Signal - Window {subplot+2}')
                axes[1].set_xlabel('Time (s)')
                axes[1].set_ylabel('Voltage (mV)')
                axes[1].legend(fontsize=8)
                axes[1].grid(alpha=0.3)
            else:
                axes[1].set_visible(False)
            plt.tight_layout()
            plt.savefig(os.path.join(save_path, f'{filename}_window_{subplot+1}_{subplot+2}.png'))
            plt.close()
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

if error_id_window_list:
    error_id_window_df = pd.DataFrame(error_id_window_list)
    error_id_window_df.to_csv(os.path.join(save_dir, 'error_id_window.csv'), index=False)
    display(error_id_window_df)